# Machine Doctor — Deep Learning Add-on
## Step 1: Download & Explore the CWRU Bearing Dataset

This is **real accelerometer data** from an actual motor test rig at Case Western Reserve University — not synthetic data. It's the standard benchmark dataset used in almost every bearing-fault deep learning paper.

**Before running:** In Colab, go to `Runtime > Change runtime type` and select **GPU** (T4 is fine, free tier).

In [ ]:
# Check we actually got a GPU
import torch
print("GPU available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None -- go to Runtime > Change runtime type > GPU")

In [ ]:
# Download the dataset from Kaggle (pre-organized, cleaned version of the
# official CWRU data -- avoids the raw .mat files' inconsistent internal
# key-naming across files, e.g. 'X097_DE_time' vs 'X098_DE_time').
!pip install -q kagglehub
import kagglehub

dataset_path = kagglehub.dataset_download("esraakhaled299/cwru-data")
print("Downloaded to:", dataset_path)

In [ ]:
# Explore the folder structure so we know exactly what we're working with
# before writing any loading code -- same "look before you leap" habit
# we used throughout the main Machine Doctor project.
import os

for root, dirs, files in os.walk(dataset_path):
    depth = root.replace(dataset_path, "").count(os.sep)
    indent = "  " * depth
    print(f"{indent}{os.path.basename(root)}/")
    if depth >= 3:  # don't spam every single file, just enough to see the pattern
        for f in files[:3]:
            print(f"{indent}  {f}")
        if len(files) > 3:
            print(f"{indent}  ... ({len(files)} files total)")

In [ ]:
# Load ONE file and actually inspect it -- confirm the data is what we
# expect (a 1D vibration signal) before building anything on top of it.
import scipy.io
import glob

# Find any .mat file to test with
sample_files = glob.glob(os.path.join(dataset_path, "**", "*.mat"), recursive=True)
print(f"Found {len(sample_files)} .mat files total")
print("Example path:", sample_files[0])

mat = scipy.io.loadmat(sample_files[0])
# .mat files load as a dict; real data keys usually contain 'DE_time' (Drive End)
data_keys = [k for k in mat.keys() if not k.startswith("__")]
print("Keys in this file:", data_keys)

de_key = [k for k in data_keys if "DE_time" in k][0]
signal = mat[de_key].flatten()
print(f"Signal shape: {signal.shape}, dtype: {signal.dtype}")
print(f"First 10 values: {signal[:10]}")

In [ ]:
# Visual sanity check: plot a healthy signal vs a faulty one side by side.
# A real fault should visibly look "spikier"/more irregular than normal --
# if we can't see ANY difference by eye, something's wrong with our loading.
import matplotlib.pyplot as plt

normal_files = [f for f in sample_files if "Normal" in f]
fault_files = [f for f in sample_files if "Ball" in f or "InnerRace" in f or "OuterRace" in f]

def load_de_signal(filepath):
    mat = scipy.io.loadmat(filepath)
    key = [k for k in mat.keys() if "DE_time" in k][0]
    return mat[key].flatten()

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

if normal_files:
    sig = load_de_signal(normal_files[0])[:2000]
    axes[0].plot(sig, linewidth=0.6, color="#2b6cb0")
    axes[0].set_title(f"Healthy bearing\n{os.path.basename(normal_files[0])}")

if fault_files:
    sig = load_de_signal(fault_files[0])[:2000]
    axes[1].plot(sig, linewidth=0.6, color="#e53e3e")
    axes[1].set_title(f"Faulty bearing\n{os.path.basename(fault_files[0])}")

plt.tight_layout()
plt.show()

print("\nIf the right (faulty) signal doesn't look visibly different from the")
print("left (healthy) one, stop here and double check the file paths above --")
print("a real fault signature should be visually obvious even before any FFT.")